# An AI Agent That Uses 2 Tools - Calculator and Weather API 


# STEPS

### Step 1) Create virtual environment (Took 2 minutes):
> C:\Users\hi\AppData\Local\Programs\Python\Python310\python.exe -m venv .venv


### Step 2) Activate virtual envir
> .venv\Scripts\Activate.ps1

### Step 3) Install jupyter (Took 3 minutes):
> python -m pip install jupyter


### Step 4) Run jupyter:
> python -m jupyter notebook


### Step 5: Install Following

In [1]:
# Following is compatible with above
!pip install langchain==0.2.14 langchain-community==0.2.12 langchain-openai==0.1.25

  Using cached langchain-0.2.14-py3-none-any.whl (997 kB)
  Using cached langchain_community-0.2.12-py3-none-any.whl (2.3 MB)
  Using cached langchain_openai-0.1.25-py3-none-any.whl (51 kB)
  Using cached langchain_core-0.2.43-py3-none-any.whl (397 kB)
  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Using cached pydantic-2.12.5-py3-none-any.whl (463 kB)
  Using cached langchain_text_splitters-0.2.4-py3-none-any.whl (25 kB)
  Using cached sqlalchemy-2.0.48-cp310-cp310-win_amd64.whl (2.1 MB)
  Using cached tenacity-8.5.0-py3-none-any.whl (28 kB)
  Using cached async_timeout-4.0.3-py3-none-any.whl (5.7 kB)
  Using cached langsmith-0.1.147-py3-none-any.whl (311 kB)
  Using cached aiohttp-3.13.3-cp310-cp310-win_amd64.whl (456 kB)
  Using cached dataclasses_json-0.6.7-py3-none-any.whl (28 kB)
  Using cached tiktoken-0.12.0-cp310-cp310-win_amd64.whl (879 kB)
  Using cached openai-1.109.1-py3-none-any.whl (948 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl (7.5 kB)
  Usi


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import sys, requests
import importlib.metadata
import openai

print("Python:", sys.version)
print("Requests:", requests.__version__)
print("OpenAI:", openai.__version__)
print("langchain:", importlib.metadata.version("langchain"))
print("langchain-community:", importlib.metadata.version("langchain-community"))
print("langchain-openai:", importlib.metadata.version("langchain-openai"))

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Requests: 2.33.1
OpenAI: 2.36.0
langchain: 1.3.6
langchain-community: 0.4.2
langchain-openai: 1.2.2


# Step6: Run the code

### Try
```
input1: What is 1 + 2 * 4
input2: What is the current weather condition in Moscow ?
input3: I am trying to calculate this math expression. What is one plus two times four ?
input4: I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?

In [2]:
# langchain_agent_calculator_weather.py
from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

import math
import requests

import os
from dotenv import load_dotenv


# 1. Load all variables from the .env file
load_dotenv()

# 2. Read specific keys securely
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")


# ==================================================
# TOOL 1 : Calculator
# ==================================================
@tool
def calculator(expression: str) -> str:
    """
    Evaluate a mathematical expression.
    Examples:
    25*4
    100/5
    sqrt(16)
    sin(0.5)
    """
    print("...Running calculator tool")

    try:
        result = eval( expression,{"__builtins__": None}, vars(math),)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"


# ==================================================
# TOOL 2 : Weather
# ==================================================
@tool
def get_weather(city: str) -> str:
    """
    Get current weather information for a city.
    """

    print("...Running weather tool")

    try:
        geo_url = (
            f"https://geocoding-api.open-meteo.com/"
            f"v1/search?name={city}"
        )

        geo_response = requests.get( geo_url, timeout=10 )

        geo_data = geo_response.json()

        if "results" not in geo_data:
            return "City not found"

        latitude = geo_data["results"][0]["latitude"]
        longitude = geo_data["results"][0]["longitude"]

        weather_url = (
            "https://api.open-meteo.com/v1/forecast"
            f"?latitude={latitude}"
            f"&longitude={longitude}"
            "&current_weather=true"
        )

        weather_response = requests.get(weather_url, timeout=10 )

        weather_data = weather_response.json()

        current = weather_data.get("current_weather")

        if current is None:
            return "Weather data unavailable"

        temperature = current["temperature"]
        windspeed   = current["windspeed"]

        return (
            f"Temperature: {temperature}°C, "
            f"Wind Speed: {windspeed} km/h"
        )

    except Exception as e:
        return f"Weather error: {e}"


def main():

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0,
        api_key=OPENAI_API_KEY,
    )

    tools = [ calculator,  get_weather, ]

    agent = create_agent(
        model=llm,
        tools=tools,
        system_prompt="""
        You are a helpful AI assistant.

        Use the calculator tool whenever mathematical
        calculations are required.

        Use the get_weather tool whenever weather
        information is requested.
        """
    )

    print("\nAgent Ready")
    print("Type 'exit' to quit\n")

    while True:

        user_input = input("Ask something: ")

        if user_input.lower() == "exit":
            break

        try:
            result = agent.invoke(
                {
                    "messages": [
                        {
                            "role": "user",
                            "content": user_input
                        }
                    ]
                }
            )

            # Extract assistant response
            assistant_message = result["messages"][-1]

            print("\nFinal Answer:")
            print(assistant_message.content)

        except Exception as e:
            print(f"\nError: {e}")

        print("-" * 60)


if __name__ == "__main__":
    main()


Agent Ready
Type 'exit' to quit



Ask something:  what is 1 + 2 * 4


...Running calculator tool

Final Answer:
The result of \( 1 + 2 \times 4 \) is 9.
------------------------------------------------------------


Ask something:  how is the weather in moscow


...Running weather tool

Final Answer:
The current weather in Moscow is 27.7°C with a wind speed of 5.4 km/h.
------------------------------------------------------------


Ask something:  exit


# Output from Trial Run

Ask something:  What is 1 + 2 * 4


> Entering new AgentExecutor chain...
Thought: I need to calculate the expression 1 + 2 * 4. According to the order of operations, I should first perform the multiplication and then the addition. 

Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```
...Running calculator tool

Observation: 9
Thought:I need to verify the calculation of the expression 1 + 2 * 4 to ensure accuracy. 

Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```

...Running calculator tool

Observation: 9
Thought:I have confirmed the calculation of the expression 1 + 2 * 4. 

Thought: I now know the final answer.
Final Answer: 9

> Finished chain.
Final Answer: 9


+++++++++++++++++

Ask something:  What is the current weather condition in Moscow ?


> Entering new AgentExecutor chain...
Thought: I need to check the current weather conditions in Moscow.
Action:
```
{
  "action": "Weather",
  "action_input": "Moscow"
}
```
...Running weather tool

Observation: Temperature: 8.7°C, Wind Speed: 6.1 km/h
Thought:I now know the final answer
Final Answer: The current weather condition in Moscow is 8.7°C with a wind speed of 6.1 km/h.

> Finished chain.
Final Answer: The current weather condition in Moscow is 8.7°C with a wind speed of 6.1 km/h.


+++++++++++++++++

Ask something:  I am trying to calculate this math expression. What is one plus two times four ?


> Entering new AgentExecutor chain...
Thought: According to the order of operations, I need to perform the multiplication first and then the addition. So, I will calculate two times four first and then add one to the result. 
Action:
```
{
  "action": "Calculator",
  "action_input": "1 + 2 * 4"
}
```
...Running calculator tool

Observation: 9
Thought:I now know the final answer.  
Final Answer: 9

> Finished chain.
Final Answer: 9


+++++++++++++++++

Ask something:  I am living in Moscow. I was thinking of going to the park. I wonder how is weather outside. Is it going going to be nice today ?


> Entering new AgentExecutor chain...
Thought: I need to check the current weather in Moscow to determine if it's nice outside for a visit to the park. 
Action:
```
{
  "action": "Weather",
  "action_input": "Moscow"
}
```
...Running weather tool

Observation: Temperature: 8.7°C, Wind Speed: 6.1 km/h
Thought:The current temperature in Moscow is 8.7°C with a wind speed of 6.1 km/h. This suggests that the weather is relatively cool, but it may still be pleasant for a visit to the park, depending on personal preferences for temperature. 

Final Answer: The weather in Moscow is currently 8.7°C with a light wind, which may be nice for a park visit if you enjoy cooler temperatures.

> Finished chain.
Final Answer: The weather in Moscow is currently 8.7°C with a light wind, which may be nice for a park visit if you enjoy cooler temperatures.


+++++++++++++++++